# W10D1 — LangGraph: Stateful Agent Graphs

## Objective
Build a stateful agent workflow using LangGraph with classification, conditional routing, response generation, and human-in-the-loop pause/resume.

## Tasks
1. Create a three-node graph: `classify → route → respond`.
2. Add conditional routing based on the classification result.
3. Test the graph with five different inputs.
4. Implement human-in-the-loop pause, human input, and resume using checkpointing.
5. Save validation results and document limitations.

## Technology Stack
- Python
- LangGraph `StateGraph`
- Typed state with `TypedDict`
- `InMemorySaver` checkpointing
- Jupyter Notebook

In [1]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python executable:", sys.executable)

for package in ["langgraph", "langchain-core"]:
    try:
        print(f"{package} version:", version(package))
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

print("StateGraph: OK")
print("START/END: OK")
print("InMemorySaver: OK")

Python executable: c:\Users\ughan\CynarisInternship\AIML_PROJECT\.venv-w9d1\Scripts\python.exe
langgraph version: 1.2.16
langchain-core version: 1.6.13
StateGraph: OK
START/END: OK
InMemorySaver: OK


In [2]:
from typing import TypedDict


class AgentState(TypedDict):
    user_input: str
    category: str
    route: str
    response: str


print("State schema defined.")
print("Fields:", list(AgentState.__annotations__.keys()))

State schema defined.
Fields: ['user_input', 'category', 'route', 'response']


In [3]:
def classify_node(state: AgentState) -> dict:
    text = state["user_input"].lower()

    if any(word in text for word in ["error", "bug", "failed", "exception"]):
        category = "technical"
    elif any(word in text for word in ["hello", "hi", "hey", "good morning"]):
        category = "greeting"
    elif any(word in text for word in ["help", "how", "what", "explain", "why"]):
        category = "question"
    else:
        category = "general"

    return {"category": category}


# Test the node independently before building the full graph.
test_state = {
    "user_input": "My Python code has an error",
    "category": "",
    "route": "",
    "response": ""
}

classification_result = classify_node(test_state)
print("Input:", test_state["user_input"])
print("Classification:", classification_result["category"])

Input: My Python code has an error
Classification: technical


In [4]:
def classify_node(state: AgentState) -> dict:
    text = state["user_input"].lower()

    if any(word in text for word in ["error", "bug", "failed", "exception"]):
        category = "technical"
    elif any(word in text for word in ["hello", "hi", "hey", "good morning"]):
        category = "greeting"
    elif any(word in text for word in ["help", "how", "what", "explain", "why"]):
        category = "question"
    else:
        category = "general"

    return {"category": category}


# Test the node independently before building the full graph.
test_state = {
    "user_input": "My Python code has an error",
    "category": "",
    "route": "",
    "response": ""
}

classification_result = classify_node(test_state)
print("Input:", test_state["user_input"])
print("Classification:", classification_result["category"])

Input: My Python code has an error
Classification: technical


In [5]:
def route_node(state: AgentState) -> dict:
    category = state["category"]

    route_map = {
        "technical": "technical_support",
        "greeting": "friendly_greeting",
        "question": "answer_question",
        "general": "general_response",
    }

    return {"route": route_map.get(category, "general_response")}


# Test routing independently.
routing_state = {
    "user_input": "My Python code has an error",
    "category": "technical",
    "route": "",
    "response": ""
}

routing_result = route_node(routing_state)
print("Category:", routing_state["category"])
print("Selected route:", routing_result["route"])

Category: technical
Selected route: technical_support


In [6]:
def respond_node(state: AgentState) -> dict:
    route = state["route"]
    user_input = state["user_input"]

    responses = {
        "technical_support": (
            f"I can help troubleshoot this technical issue: {user_input}"
        ),
        "friendly_greeting": (
            "Hello! 👋 How can I help you today?"
        ),
        "answer_question": (
            f"Let's work through your question: {user_input}"
        ),
        "general_response": (
            f"Thanks for sharing. Tell me a little more about: {user_input}"
        ),
    }

    return {"response": responses.get(
        route, "I'm ready to help. Please provide more details."
    )}


# Test the response node independently.
response_state = {
    "user_input": "My Python code has an error",
    "category": "technical",
    "route": "technical_support",
    "response": ""
}

response_result = respond_node(response_state)
print("Generated response:")
print(response_result["response"])

Generated response:
I can help troubleshoot this technical issue: My Python code has an error


In [7]:
from langgraph.graph import StateGraph, START, END


def build_agent_graph():
    builder = StateGraph(AgentState)

    # Register the three nodes.
    builder.add_node("classify", classify_node)
    builder.add_node("route", route_node)
    builder.add_node("respond", respond_node)

    # Define the graph flow.
    builder.add_edge(START, "classify")
    builder.add_edge("classify", "route")
    builder.add_conditional_edges(
        "route",
        lambda state: state["route"],
        {
            "technical_support": "respond",
            "friendly_greeting": "respond",
            "answer_question": "respond",
            "general_response": "respond",
        },
    )
    builder.add_edge("respond", END)

    return builder.compile()


agent_graph = build_agent_graph()

print("LangGraph compiled successfully.")
print("Nodes:", list(agent_graph.nodes.keys()))

LangGraph compiled successfully.
Nodes: ['__start__', 'classify', 'route', 'respond']


In [8]:
test_inputs = [
    "Hello there!",
    "My Python code has an error",
    "How does a robot use computer vision?",
    "Explain LangGraph to me",
    "Today I worked on my project",
]

test_results = []

for user_input in test_inputs:
    result = agent_graph.invoke({
        "user_input": user_input,
        "category": "",
        "route": "",
        "response": "",
    })

    test_results.append({
        "input": user_input,
        "category": result["category"],
        "route": result["route"],
        "response": result["response"],
    })

for index, item in enumerate(test_results, start=1):
    print(f"\nTest {index}")
    print("Input:", item["input"])
    print("Category:", item["category"])
    print("Route:", item["route"])
    print("Response:", item["response"])

print(f"\nTotal tests executed: {len(test_results)}")


Test 1
Input: Hello there!
Category: greeting
Route: friendly_greeting
Response: Hello! 👋 How can I help you today?

Test 2
Input: My Python code has an error
Category: technical
Route: technical_support
Response: I can help troubleshoot this technical issue: My Python code has an error

Test 3
Input: How does a robot use computer vision?
Category: question
Route: answer_question
Response: Let's work through your question: How does a robot use computer vision?

Test 4
Input: Explain LangGraph to me
Category: question
Route: answer_question
Response: Let's work through your question: Explain LangGraph to me

Test 5
Input: Today I worked on my project
Category: general
Route: general_response
Response: Thanks for sharing. Tell me a little more about: Today I worked on my project

Total tests executed: 5


In [9]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


def human_review_node(state: AgentState) -> dict:
    decision = interrupt({
        "message": "Human review required",
        "draft_response": state["response"],
        "question": "Approve this response or provide a replacement?",
    })

    if isinstance(decision, dict):
        final_response = decision.get(
            "response", state["response"]
        )
    else:
        final_response = str(decision)

    return {"response": final_response}


def build_human_review_graph():
    builder = StateGraph(AgentState)

    builder.add_node("classify", classify_node)
    builder.add_node("route", route_node)
    builder.add_node("respond", respond_node)
    builder.add_node("human_review", human_review_node)

    builder.add_edge(START, "classify")
    builder.add_edge("classify", "route")
    builder.add_conditional_edges(
        "route",
        lambda state: state["route"],
        {
            "technical_support": "respond",
            "friendly_greeting": "respond",
            "answer_question": "respond",
            "general_response": "respond",
        },
    )
    builder.add_edge("respond", "human_review")
    builder.add_edge("human_review", END)

    return builder.compile(checkpointer=InMemorySaver())


human_review_graph = build_human_review_graph()
print("Human-review graph compiled:", human_review_graph is not None)

Human-review graph compiled: True


In [10]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


def human_review_node(state: AgentState) -> dict:
    decision = interrupt({
        "message": "Human review required",
        "draft_response": state["response"],
        "question": "Approve this response or provide a replacement?",
    })

    if isinstance(decision, dict):
        final_response = decision.get(
            "response", state["response"]
        )
    else:
        final_response = str(decision)

    return {"response": final_response}


def build_human_review_graph():
    builder = StateGraph(AgentState)

    builder.add_node("classify", classify_node)
    builder.add_node("route", route_node)
    builder.add_node("respond", respond_node)
    builder.add_node("human_review", human_review_node)

    builder.add_edge(START, "classify")
    builder.add_edge("classify", "route")
    builder.add_conditional_edges(
        "route",
        lambda state: state["route"],
        {
            "technical_support": "respond",
            "friendly_greeting": "respond",
            "answer_question": "respond",
            "general_response": "respond",
        },
    )
    builder.add_edge("respond", "human_review")
    builder.add_edge("human_review", END)

    return builder.compile(checkpointer=InMemorySaver())


human_review_graph = build_human_review_graph()
print("Human-review graph compiled:", human_review_graph is not None)

Human-review graph compiled: True


In [11]:
from langgraph.types import Command

review_config = {
    "configurable": {
        "thread_id": "w10d1-human-review-1"
    }
}

initial_state = {
    "user_input": "How does a robot use computer vision?",
    "category": "",
    "route": "",
    "response": "",
}

paused_result = human_review_graph.invoke(
    initial_state,
    config=review_config,
)

print("Graph paused for human review.")
print("Interrupt present:", bool(paused_result.get("__interrupt__")))

if paused_result.get("__interrupt__"):
    print("Review request:")
    print(paused_result["__interrupt__"][0].value)

Graph paused for human review.
Interrupt present: True
Review request:
{'message': 'Human review required', 'draft_response': "Let's work through your question: How does a robot use computer vision?", 'question': 'Approve this response or provide a replacement?'}


In [12]:
human_decision = {
    "response": (
        "Computer vision helps robots interpret camera images to detect "
        "objects, estimate positions, and navigate their environment."
    )
}

resumed_result = human_review_graph.invoke(
    Command(resume=human_decision),
    config=review_config,
)

print("Graph resumed successfully.")
print("Final response:")
print(resumed_result["response"])
print("Interrupt still pending:", bool(resumed_result.get("__interrupt__")))

Graph resumed successfully.
Final response:
Computer vision helps robots interpret camera images to detect objects, estimate positions, and navigate their environment.
Interrupt still pending: False


In [13]:
expected_categories = [
    "greeting",
    "technical",
    "question",
    "question",
    "general",
]

test_summary = []

for index, (item, expected) in enumerate(
    zip(test_results, expected_categories), start=1
):
    passed = (
        item["category"] == expected
        and bool(item["route"])
        and bool(item["response"])
    )

    test_summary.append({
        "test": index,
        "expected_category": expected,
        "actual_category": item["category"],
        "route": item["route"],
        "passed": passed,
    })

for result in test_summary:
    status = "PASS" if result["passed"] else "FAIL"
    print(
        f"Test {result['test']}: {status} | "
        f"Expected={result['expected_category']} | "
        f"Actual={result['actual_category']} | "
        f"Route={result['route']}"
    )

passed_count = sum(item["passed"] for item in test_summary)
print(f"\nResult: {passed_count}/{len(test_summary)} tests passed")

Test 1: PASS | Expected=greeting | Actual=greeting | Route=friendly_greeting
Test 2: PASS | Expected=technical | Actual=technical | Route=technical_support
Test 3: PASS | Expected=question | Actual=question | Route=answer_question
Test 4: PASS | Expected=question | Actual=question | Route=answer_question
Test 5: PASS | Expected=general | Actual=general | Route=general_response

Result: 5/5 tests passed


In [14]:
import json
from pathlib import Path
from datetime import datetime, timezone

output_dir = Path("../outputs/W10D1")
output_dir.mkdir(parents=True, exist_ok=True)

evaluation = {
    "day": "W10D1",
    "project": "LangGraph Stateful Agent Graphs",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "graph_nodes": ["classify", "route", "respond"],
    "conditional_routing": True,
    "five_input_tests": {
        "passed": passed_count,
        "total": len(test_summary),
        "results": test_summary,
    },
    "human_in_the_loop": {
        "pause_detected": bool(paused_result.get("__interrupt__")),
        "resumed_successfully": not bool(
            resumed_result.get("__interrupt__")
        ),
        "human_response_applied": (
            resumed_result["response"] == human_decision["response"]
        ),
    },
    "limitations": [
        "Classification uses keyword rules, not an LLM.",
        "The human-review checkpoint uses in-memory storage and is not persistent across process restarts.",
        "Test inputs cover a small set of predefined examples."
    ],
}

evaluation_path = output_dir / "evaluation_results.json"
evaluation_path.write_text(
    json.dumps(evaluation, indent=2),
    encoding="utf-8",
)

print("Evaluation saved:", evaluation_path)
print("Five-input tests:", f"{passed_count}/{len(test_summary)}")
print(
    "Human pause/resume:",
    "PASS" if (
        evaluation["human_in_the_loop"]["pause_detected"]
        and evaluation["human_in_the_loop"]["resumed_successfully"]
        and evaluation["human_in_the_loop"]["human_response_applied"]
    ) else "FAIL"
)

Evaluation saved: ..\outputs\W10D1\evaluation_results.json
Five-input tests: 5/5
Human pause/resume: PASS


In [15]:
import json
from pathlib import Path

evaluation_path = Path("../outputs/W10D1/evaluation_results.json")

saved_evaluation = json.loads(
    evaluation_path.read_text(encoding="utf-8")
)

assert saved_evaluation["five_input_tests"]["passed"] == 5
assert saved_evaluation["five_input_tests"]["total"] == 5
assert saved_evaluation["human_in_the_loop"]["pause_detected"] is True
assert saved_evaluation["human_in_the_loop"]["resumed_successfully"] is True
assert saved_evaluation["human_in_the_loop"]["human_response_applied"] is True

print("JSON file parsed successfully.")
print("Five-input evaluation: PASS")
print("Human-in-the-loop evaluation: PASS")
print("All evidence assertions passed.")

JSON file parsed successfully.
Five-input evaluation: PASS
Human-in-the-loop evaluation: PASS
All evidence assertions passed.


In [16]:
from pathlib import Path

validation_text = """# W10D1 Validation — LangGraph Stateful Agent Graphs

## Implementation
- Created a three-node LangGraph workflow: classify, route, and respond.
- Added conditional routing based on classification.
- Used LangGraph InMemorySaver checkpointing for human review.
- Implemented interrupt, human input, and resume.

## Test Results
- Five predefined inputs tested: 5/5 passed.
- Greeting route: passed.
- Technical-support route: passed.
- Question route: passed.
- General-response route: passed.
- Human-review pause: passed.
- Human-review resume: passed.
- Saved evaluation JSON parsed and all evidence assertions passed.

## Evidence
- `outputs/W10D1/evaluation_results.json`
- `notebooks/W10D1_langgraph_stateful_agent_graphs.ipynb`

## Limitations
- Classification uses keyword rules rather than an LLM.
- In-memory checkpoints do not persist across process restarts.
- Five examples are a small test set and do not establish broad classification accuracy.
"""

validation_path = Path("../outputs/W10D1/VALIDATION.md")
validation_path.write_text(validation_text, encoding="utf-8")

print("Validation document saved:", validation_path)
print("Document size:", validation_path.stat().st_size, "bytes")
print("Contains test results:", "5/5 passed" in validation_text)
print("Contains limitations:", "## Limitations" in validation_text)

Validation document saved: ..\outputs\W10D1\VALIDATION.md
Document size: 988 bytes
Contains test results: True
Contains limitations: True


In [17]:
from pathlib import Path

validation_path = Path("../outputs/W10D1/VALIDATION.md")
validation = validation_path.read_text(encoding="utf-8")

required_sections = [
    "# W10D1 Validation",
    "## Implementation",
    "## Test Results",
    "## Evidence",
    "## Limitations",
]

for section in required_sections:
    status = "PASS" if section in validation else "FAIL"
    print(f"{status}: {section}")

assert all(section in validation for section in required_sections)
print("\nValidation document check: PASS")

PASS: # W10D1 Validation
PASS: ## Implementation
PASS: ## Test Results
PASS: ## Evidence
PASS: ## Limitations

Validation document check: PASS
